This tutorial is a quick demonstration of how to build a species distribution
model (SDM) for Eurasian Red Squirrel (_Sciurus vulgaris_) using Maximum Entropy
(MaxEnt). MaxEnt is a popular method for constructing SDMs for presence-only 
data. The following should not be taken as a "best practice" example, as it only
aims to outline some of the key steps in the modelling process without applying
any of the rigour required to build a fit for purpose model.

## Software setup
To fit distribution models to FinBIF occurrence records will require the
installation of some additional software.

The `{raster}` package is needed for working with gridded data.

In [ ]:
if (!require("raster")) install.packages("raster")
library(raster)

The `{sf}` package is needed for working with spatial data.

In [ ]:
if (!require("sf")) install.packages("sf")
library(sf)

The `{geodata}` package is needed for getting bioclimatic data from WordlClim.

In [ ]:
if (!require("geodata")) install.packages("geodata")
library(geodata)

The `{rnaturalearth}` package is needed for getting country borders.

In [ ]:
if (!require("rnaturalearth")) install.packages("rnaturalearth")
library(rnaturalearth)

The `{maxnet}` package is an implementation of the popular presence-only
modelling software, MaxEnt, using the elastic-net regularised generalised linear
models (glmnet).

In [ ]:
if (!require("maxnet")) install.packages("maxnet")
library(maxnet)

To obtain the occurrence records to model the Eurasian Red Squirrel distribution
will require the `{finbif}` package

In [ ]:
library(finbif)

For the sake of reproducibility it is a good idea to set an explicit path to
hold cached data.

In [ ]:
cache_path <- "cache"
dir.create(cache_path)
options(finbif_cache_path = cache_path)

## Occurrence & background
Fitting a presence-only SDM requires two inputs: occurrence records and
background data points to compare with the occurrence records. Both occurrence
and background points can be downloaded directly from FinBIF.

In practice careful thought should go into the selection of both data types.
Here the details of selecting both are glossed over for the sake of brevity and
demonstration.

### Filtering
A common set of filters can be used for both occurrence records and background
data. This filter limits records to those within a bounding box encompassing
(approximately) Finland, and restricts the data to those with a maximum
coordinate uncertainty of 100m.

In [ ]:
coordinates_filter <- list(
  coordinates = list(lat = c(60, 70), lon = c(20, 30), system = "wgs84"),
  coordinates_uncertainty_max = 100
)

## Getting a FinBIF access token

To use the FinBIF API you must first request and set a personal access token. You can request an API token to be sent to your email address with the function finbif_get_token().


In [ ]:
finbif_request_token("your@email.com")

Sys.setenv(
  FINBIF_ACCESS_TOKEN = "access_token_sent_to_your_email"
)
# Note: the above is not a real access token. Do not try using it.

Copy the access token that was sent to your email and set it as the environment variable FINBIF_ACCESS_TOKEN either for the current session,

or by adding it to a Renviron startup file (see here for details).

### Occurrence records
Using the predetermined filter the coordinates of the most recent (up to) 5000
occurrences of Eurasian Red Squirrel can be downloaded from FinBIF.

In [ ]:
red_squirrel_points <- finbif_occurrence(
  species = "Sciurus vulgaris",
  filter  = coordinates_filter,
  select  = c("lon_wgs84", "lat_wgs84"),
  n       = 5e3
)

### Background points
Using the same filter we can obtain background points by selecting a random
sample (10,000) of mammal occurrences. Using the records of a higher taxonomic
group rather than selecting points completely at random helps to mitigate
sampling bias in the occurrence data on the assumption that bias will be similar
in the Squirrel occurrences to the bias in the background points chosen in this
way. In reality this is only one source of bias and more careful thought is
needed to minimise the effect of bias in the model data.

In [ ]:
background_points <- finbif_occurrence(
  class  = "Mammals",
  filter = coordinates_filter,
  select = c("lon_wgs84", "lat_wgs84"),
  n      = 1e4,
  sample = TRUE
)

### Plotting occurrences
It is useful to compare the Squirrel occurrences to the background points
graphically.

First plot the Finnish border and background data, selecting a semi-transparent colour to convey
the density of points. Alsom overlay the Eurasian Red Squirrel occurrence points in a different shade.

In [ ]:
finland_map <- ne_countries(country = c("Finland", "Aland"))$geom
plot(finland_map)
points(
  x   = background_points,
  col = rgb(0, 0, 0, .1)
)
points(
  x   = red_squirrel_points,
  col = rgb(1, 0, 0, .1),
  pch = 19,
  cex = .2
)

## Climate data 
You can obtain climate layers directly from [WorldClim](http://worldclim.org)
with the `{geodata}` package function `worldclim_tile()`. We use these data
purely for convenience, and there is little guarantee that all, or any of them,
is related to a given taxon's distribution. A more robust and rigorous model 
requires a deeper understanding of the underlying drivers of the taxon of 
interest's distribution to select appropriate response variables.

### Getting climate data
Specify the appropriate latitude and longitude to retrieve the tiles that cover
the majority of Finland. There are 19 bioclimatic variables available, the
following retrieves them all and stores them in the same cache directory as the
FinBIF data.

In [ ]:
climate_high_res <- worldclim_tile(
  var  = "bio",
  lat  = 65,
  lon  = 25,
  path = cache_path
)



### Extracting climate data
You can extract the climate data for all 19 variables at each of the Squirrel
occurrence locations using the `{raster}` function `extract()`.

In [ ]:
climate_red_squirrel <- extract(
  x = climate_high_res,
  y = as.data.frame(red_squirrel_points)
)

Do the same for the background data points.

In [ ]:
climate_background <- extract(
  x = climate_high_res,
  y = as.data.frame(background_points)
)

## Formatting input data
Now combine the two datasets into a single object with an indicator variable `p`
that is `1` for presence (occurrence records) and `0` for background points.

In [ ]:
input_data <- rbind(
  cbind(p = 1, climate_red_squirrel),
  cbind(p = 0, climate_background)
)

To fit a model the input data must be in the form of a `data.frame` and cannot
contain any `NA` data.

In [ ]:
input_data <- na.omit(data.frame(input_data))

## Fitting MaxEnt model
To fit the SDM use the `{maxnet}` function `maxnet()`. Here the default settings
are used. In general this is inadvisable and care should be taken to find the
appropriate model form and settings for the purpose of the model.

In [ ]:
red_squirrel_model <- maxnet(p = input_data[, 1], data = input_data[, -1:-2])

## Projection
Using the fitted model we can project the predicted distribution to the broader
region. A caveat is that projections are not reliable if they are being made to
novel parts of the climatic space.

### Climate data
In this case, it will be unnecessary to project to the high resolution climate
data. For the purpose of visualization the 2.5 arc-minutes is fine-grained
enough.

In [ ]:
climate_low_res <- worldclim_country(
  country = "Finland",
  var     = "bio",
  res     = 2.5, 
  path    = cache_path
)

However, this lower resolution data is untiled so needs to be cropped; in this
case, to the bounding box of the `{finbif}` package's internal map of Finland.

In [ ]:
climate_low_res <- crop(
  x = climate_low_res,
  y = extent(st_bbox(finland_map))
)

To make projections to these climate layers with the model requires the data as
a `data.frame`.

In [ ]:
climate_low_res_df <- as.data.frame(climate_low_res)

And also requires that the names of the bioclimatic layers are the same as the
the lower resolution version

In [ ]:
names(climate_low_res_df) <- sub("wc2\\.1_30s_", "", names(climate_low_res_df))

(the `"_06"` suffix indicates the sixth Worldclim tile that covers Finland).

### Projecting model
Projecting to the new climate data can be done with `{maxnet}` package's
`predict` method and selecting the "cloglog" (complementary log-log) output
type.

In [ ]:
predicted_dist <- predict(
  object   = red_squirrel_model,
  newdata  = climate_low_res_df,
  type     = "cloglog"
)

An empty gridded data layer can be created using the low-res climate layers as
a template.

In [ ]:
predicted_dist_raster <- raster(climate_low_res$wc2.1_30s_bio_1)

And the projected Squirrel distribution can be added by using non-NA data cells
from the climate layers as an index.

In [ ]:
predicted_dist_raster[!is.nan(predicted_dist_raster[])] <- predicted_dist

### Plotting projection
A plot of the Eurasian Red Squirrel's modelled distribution can be made using
the `plot` method from the `{raster}` package, and finally adding the original Squirrel occurrence data.

In [ ]:
plot(
  x   = predicted_dist_raster,
  col = hcl.colors(12),
  las = 1,
  asp = 2.4
)

points(
  x   = as.data.frame(red_squirrel_points),
  cex = .05,
  col = rgb(1, 0, 0, .05),
  pch = 19
)